## ATL06 Tiled Product Demonstration

In [ ]:
from sliderule import sliderule, s2tile
import geopandas as gpd
import matplotlib.pyplot as plt
import duckdb

### Region Covered

In [ ]:
antarctica_poly = [
    {"lon": -151.86,    "lat": -87.51},
    {"lon": -136.64,    "lat": -86.99},
    {"lon": -146.35,    "lat": -86.27},
    {"lon": -159.34,    "lat": -86.68},
    {"lon": -151.86,    "lat": -87.51}
]

### Define Areas of Interest

In [ ]:
antarctica_poly_50km2 = [
    {"lon": -145.6740999746769,     "lat": -86.77017082487883},
    {"lon": -144.71490455261718,    "lat": -86.73238150176192},
    {"lon": -145.32664951627441,    "lat": -86.68216031925881},
    {"lon": -146.2780643571965,     "lat": -86.7193698909303},
    {"lon": -145.6740999746769,     "lat": -86.77017082487883}
]
antarctica_poly_300km2 = [
    {"lon": -147.06343045985406,    "lat": -86.83690411200126},
    {"lon": -144.79209365113724,    "lat": -86.75097547087802},
    {"lon": -146.33930539572938,    "lat": -86.62049627004224},
    {"lon": -148.55905432356855,    "lat": -86.70301545138857},
    {"lon": -147.06343045985406,    "lat": -86.83690411200126}
]
antarctica_convex_poly_450km2 = [
    {"lon": -148.49117484589732,    "lat": -86.65536364330114},
    {"lon": -149.8648748595424,     "lat": -86.75211342612057},
    {"lon": -149.51467388586605,    "lat": -86.83547573017331},
    {"lon": -147.6964317971983,     "lat": -86.82948929703882},
    {"lon": -146.19497874424465,    "lat": -86.77060503864796},
    {"lon": -145.71920973197888,    "lat": -86.68789287135104},
    {"lon": -146.43847873389288,    "lat": -86.58801848073247},
    {"lon": -148.49117484589732,    "lat": -86.65536364330114}
]
antarctica_poly_5000km2 = [
    {"lon": -148.6160509646996,     "lat": -87.33643108138267},
    {"lon": -137.9164826505549,     "lat": -86.9363391783649},
    {"lon": -144.55925487296605,    "lat": -86.4592390675824},
    {"lon": -154.32249446054084,    "lat": -86.79901992663235},
    {"lon": -148.6160509646996,     "lat": -87.33643108138267}
]
aois = [antarctica_poly_50km2, antarctica_poly_300km2, antarctica_convex_poly_450km2, antarctica_poly_5000km2]
names = ["Bbox50km2", "Bbox300km2", "Convex450km2", "Bbox5000km2", "coverage"]
regions = {names[i]: sliderule.toregion(aois[i]) for i in range(len(aois))} | {"coverage": sliderule.toregion(antarctica_poly)}

### Display Areas of Interest

In [ ]:
# Antarctic Polar Stereographic (South)
crs = "EPSG:3031"
countries = gpd.read_file("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")
antarctica = countries[countries["ADMIN"] == "Antarctica"].to_crs(crs)
antarctica_pc = countries[countries["ADMIN"] == "Antarctica"].to_crs("EPSG:4326")

labels = names
colors = ["tab:red", "tab:orange", "tab:blue", "tab:green", "brown"]
region_gdfs = [r["gdf"].set_crs("EPSG:4326", allow_override=True).to_crs(crs) for r in regions.values()]

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(14, 8))

# All level 7 S2 cells that touch Antarctica (full cells, so they extend into the ocean)
S2_LEVEL = 7
con = duckdb.connect()
try:
    con.sql("LOAD geography;")
except duckdb.Error:
    con.sql("INSTALL geography FROM community; LOAD geography;")
cells = con.execute(f"""
    SELECT s2_cell_token(c) AS token, s2_astext(c::GEOGRAPHY) AS wkt
    FROM (SELECT UNNEST(s2_covering_fixed_level(s2_geogfromtext(?), {S2_LEVEL})) AS c)
""", [antarctica_pc.union_all().wkt]).df()
con.close()

# No segmentize here: some cells cross the antimeridian, and interpolating in lon/lat would
# wrap them around the globe. Level 7 cells are small enough that straight edges in 3031 look fine.
cell_geoms = gpd.GeoSeries.from_wkt(cells["wkt"], crs="EPSG:4326")
cells_gdf = gpd.GeoDataFrame(cells[["token"]], geometry=cell_geoms).to_crs(crs)

# Full Antarctica view (regions are small, so mark them with an outline of the largest)
antarctica.plot(ax=ax_full, color="whitesmoke", edgecolor="black", linewidth=0.5)
for gdf, color, label in zip(region_gdfs, colors, labels):
    gdf.boundary.plot(ax=ax_full, color=color, linewidth=1.5, label=label)
cells_gdf.boundary.plot(ax=ax_full, color="red", linewidth=0.1)
ax_full.set_title("Antarctica")
ax_full.set_axis_off()
ax_full.legend(loc="lower right")

# Zoomed view around the regions
antarctica.plot(ax=ax_zoom, color="whitesmoke", edgecolor="black", linewidth=0.5)
for gdf, color, label in zip(region_gdfs, colors, labels):
    gdf.plot(ax=ax_zoom, facecolor=color, edgecolor=color, alpha=0.3)
    gdf.boundary.plot(ax=ax_zoom, color=color, linewidth=1.5, label=label)
xmin, ymin, xmax, ymax = region_gdfs[len(labels) - 1].total_bounds
pad = 0.25 * max(xmax - xmin, ymax - ymin)
ax_zoom.set_xlim(xmin - pad, xmax + pad)
ax_zoom.set_ylim(ymin - pad, ymax + pad)
ax_zoom.set_title("Regions (zoomed)")
ax_zoom.set_axis_off()
ax_zoom.legend(loc="lower right")

# Box on the full map showing the zoom extent
ax_full.add_patch(plt.Rectangle((xmin - pad, ymin - pad), xmax - xmin + 2 * pad, ymax - ymin + 2 * pad,
                                fill=False, edgecolor="black", linestyle="--"))

plt.tight_layout()
plt.show()

### Tile Statistics

In [ ]:
BUCKET = "sliderule-public"
PREFIX = "atl06tv3"

In [ ]:
import struct
import s3fs
import pandas as pd
import pyarrow.parquet as pq
from concurrent.futures import ThreadPoolExecutor

# Listing the bucket requires credentials (anonymous access is denied)
fs = s3fs.S3FileSystem()
tile_files = [f for f in fs.find(f"{BUCKET}/{PREFIX}/tiles", detail=True).values() if f["name"].endswith(".parquet")]

def tile_stats(info):
    path = info["name"]
    with fs.open(path, "rb", block_size=1024 * 1024) as f:
        # Parquet files end with a 4-byte little-endian footer length followed by "PAR1"
        f.seek(-8, 2)
        footer_len, magic = struct.unpack("<I4s", f.read(8))
        if magic != b"PAR1":
            raise ValueError(f"{path} is not a parquet file")
        f.seek(0)
        md = pq.ParquetFile(f).metadata
    return {
        "tile": path[len(f"{BUCKET}/{PREFIX}/tiles/"):],
        "size_mb": info["size"] / (1024 * 1024),
        "footer_kb": footer_len / 1024,
        "num_rows": md.num_rows,
        "num_row_groups": md.num_row_groups,
    }

with ThreadPoolExecutor(max_workers=32) as pool:
    tiles = pd.DataFrame(pool.map(tile_stats, tile_files)).sort_values("tile", ignore_index=True)

print(f"{len(tiles)} tiles, {tiles['size_mb'].sum() / 1024:.2f} GB, {tiles['num_rows'].sum():,} rows")
tiles

In [ ]:
reader = s2tile.Reader(BUCKET, PREFIX)
coverage = reader.read_tiles(antarctica_convex_poly_450km2, names_only=True)
tiles_read = [f"ATL06T_S{tile:016X}.parquet" for tile in coverage]
tiles_read

In [ ]:
#tiles_to_stat = tiles[tiles["tile"].isin(tiles_read)]
tiles_to_stat = tiles
print(f"{'size (MB)':>10} {'footer (KB)':>12} {'rows':>14} {'row groups':>11}  tile")
for t in tiles_to_stat.sort_values("size_mb", ascending=False).itertuples():
    print(f"{t.size_mb:>10,.0f} {t.footer_kb:>12,.0f} {t.num_rows:>14,} {t.num_row_groups:>11,}  {t.tile}")